# Course 5: Winning Online AI Competitions
## Session 7: Image competitions: resolution and test-time augmentation

**Goal:** train a small CNN at two resolutions, then find which test-time augmentations help and which ones hurt.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. Data
The 8x8 digits from scikit-learn stand in for a handwritten-character competition such as Bengali.AI graphemes. The lessons about resolution and augmentation carry over; the numbers do not.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.manual_seed(0)
d = load_digits(); X = torch.tensor(d.images / 16.0, dtype=torch.float32)[:, None]; y = torch.tensor(d.target)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

### 2. The same network at two resolutions
Resolution is usually the first knob to turn in an image competition: more pixels, more detail, more compute. Here the 'low' setting is 4x4 (downsampled) and the 'high' one is the original 8x8.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, c=24):
        super().__init__(); self.c1, self.c2, self.fc = nn.Conv2d(1, c, 3, padding=1), nn.Conv2d(c, c, 3, padding=1), nn.Linear(c, 10)
    def forward(self, x): return self.fc(F.relu(self.c2(F.relu(self.c1(x)))).amax((2, 3)))

def train_model(Xa, ya, epochs=40, augment=None, seed=0):
    torch.manual_seed(seed); m = SmallCNN(); opt = torch.optim.Adam(m.parameters(), lr=3e-3)
    for _ in range(epochs):
        for idx in torch.randperm(len(Xa)).split(64):
            xb = Xa[idx] if augment is None else augment(Xa[idx])
            loss = F.cross_entropy(m(xb), ya[idx]); opt.zero_grad(); loss.backward(); opt.step()
    return m.eval()
acc = lambda m, Xa: (m(Xa).argmax(1) == yte).float().mean().item()
low = lambda t: F.avg_pool2d(t, 2)
m_low, m_high = train_model(low(Xtr), ytr), train_model(Xtr, ytr)
with torch.no_grad(): a_low, a_high = acc(m_low, low(Xte)), acc(m_high, Xte)
print(f"4x4 input: test accuracy {a_low:.3f} | 8x8 input: test accuracy {a_high:.3f}")
assert a_high > a_low

### 3. Training-time augmentation
Small random shifts create new plausible digits. The model sees more variety without new labels.

In [ ]:
def random_shift(x):
    out = torch.zeros_like(x)
    for i in range(len(x)):
        dx, dy = np.random.randint(-1, 2), np.random.randint(-1, 2)
        out[i] = torch.roll(x[i], (dy, dx), (1, 2))
    return out
np.random.seed(0)
m_aug = train_model(Xtr, ytr, augment=random_shift)
with torch.no_grad(): a_aug = acc(m_aug, Xte)
print(f"8x8 with shift augmentation in training: test accuracy {a_aug:.3f} (without: {a_high:.3f})")

Check the two numbers. On these tiny centred images a one-pixel shift can push a stroke off the edge, so augmentation may not help here, and it can cost a little. On larger images with a margin it usually helps. Augmentation is a hypothesis about the data and has to be measured like any other.

### 4. Test-time augmentation: average predictions over several views
A TTA is safe when the transformed image still has the same label and looks like something the model saw in training.

In [ ]:
@torch.no_grad()
def tta_accuracy(model, views):
    probs = sum(torch.softmax(model(v(Xte)), 1) for v in views) / len(views)
    return (probs.argmax(1) == yte).float().mean().item()

identity = lambda t: t
shifts = [identity] + [lambda t, s=s: torch.roll(t, s, (2, 3)) for s in [(0, 1), (0, -1), (1, 0), (-1, 0)]]
flip_h = lambda t: torch.flip(t, (3,)); rot180 = lambda t: torch.rot90(t, 2, (2, 3))
rows = {"no TTA": tta_accuracy(m_aug, [identity]), "5 small shifts (safe)": tta_accuracy(m_aug, shifts),
        "+ horizontal flip (harmful)": tta_accuracy(m_aug, [identity, flip_h]), "+ rotation by 180 (harmful)": tta_accuracy(m_aug, [identity, rot180])}
for k, v in rows.items(): print(f"{k:30s} test accuracy {v:.3f}")
assert rows["+ rotation by 180 (harmful)"] < rows["no TTA"] and rows["5 small shifts (safe)"] >= rows["no TTA"] - 0.01

Flips and half-turns are standard for photographs of cats. For characters they are wrong: a flipped digit is not a digit, and a 6 turned upside down is a 9. Bengali graphemes are the same. Choose augmentations by asking whether the label survives.

### 5. Check which classes a TTA damages

In [ ]:
with torch.no_grad():
    p0 = torch.softmax(m_aug(Xte), 1).argmax(1); p1 = ((torch.softmax(m_aug(Xte), 1) + torch.softmax(m_aug(rot180(Xte)), 1)) / 2).argmax(1)
per_class = [(c, ((p0 == yte) & (yte == c)).float().sum().item() / (yte == c).sum().item(), ((p1 == yte) & (yte == c)).float().sum().item() / (yte == c).sum().item()) for c in range(10)]
worst = sorted(per_class, key=lambda r: r[2] - r[1])[:3]
for c, a, b in worst: print(f"digit {c}: accuracy {a:.2f} -> {b:.2f} with the 180-degree view")

### 6. The real pipeline with timm (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU, `pip install timm albumentations`, and the Bengali.AI Handwritten Grapheme Classification data from Kaggle.
import timm, torch
# The task has three targets per image: grapheme root (168 classes), vowel diacritic (11) and consonant diacritic (7).
class GraphemeNet(torch.nn.Module):
    def __init__(self, name="resnet34"):
        super().__init__()
        self.body = timm.create_model(name, pretrained=True, in_chans=1, num_classes=0)       # check the model name with timm.list_models()
        d = self.body.num_features
        self.root, self.vowel, self.cons = torch.nn.Linear(d, 168), torch.nn.Linear(d, 11), torch.nn.Linear(d, 7)
    def forward(self, x):
        f = self.body(x); return self.root(f), self.vowel(f), self.cons(f)
# Plan:
# 1. Group folds so that validation contains grapheme combinations not seen in training (the test set did).
# 2. Train at 128x128 and at 224x224 with the same recipe; report the competition metric (weighted macro recall) for both.
# 3. Augment with small shifts, scale, slight rotation and cutout. No flips.
# 4. TTA with 3 slightly shifted or scaled views. Measure it on the validation folds before using it.

### TODO
1. Train at 16x16 (upsample the digits). Does more resolution than the data contains help?
2. Add rotation by ±10 degrees to training augmentation and to TTA. Measure each separately.
3. In the grapheme task, why do random row-wise folds overestimate the score? What should the groups be?